# Multi-Property Document Routing and SFTP Automation\n\nPublic, anonymized portfolio version of a Python/Google Drive/SFTP workflow.\n\n**Pipeline:** monthly PDF discovery → document classification → upload-history comparison → property-specific filename generation → secure SFTP upload → audit-log update.\n\nAll client-specific server addresses, usernames, key locations, local paths, and saved outputs have been removed. No actual client documents are included.\n\n**Important:** This notebook is a code showcase and requires private environment configuration, authorized access to client data, and validated SSH host keys to run. Do not publish real client files or configuration workbooks.\n

In [ ]:
# Configuration for the public demonstration
# Provide paths and credentials securely through your runtime environment.
# Never commit a private key, client documents, or live connection details.
import os
CONFIG_PATH = os.environ['CONSORCIOS_CONFIG_PATH']
UPLOAD_LOG_PATH = os.environ['CONSORCIOS_UPLOAD_LOG_PATH']


In [ ]:
# monto el gdrive y recorro todos los archivos de todas las carpetas y almaceno cuando encuentro el mes_proceso
mes_proceso = input('Ingrese año y mes de corrida en formato YYYY MM')
# mes_proceso = '2024 03'
import os
from google.colab import drive
from google.colab import files
import pandas as pd
import numpy as np
drive.mount('/content/gdrive')
#si cambia de usuario hay que cambiar estas carpetas
consorcios = pd.read_excel(CONFIG_PATH)

directory = pd.DataFrame(columns = ['URL','Folder', 'Files','Id_Consorcio'])
for nro_consorcio in consorcios.index:
  url = consorcios['URL_2'][nro_consorcio]
  id_consorcio_tabla = consorcios['Id_consorcio'][nro_consorcio]
  for foldername, _, filename in os.walk(url):
    directory.loc[len(directory)] = [url, foldername, filename,id_consorcio_tabla]

archivos_despivoteados = pd.DataFrame (columns = ['URL','Folder', 'File', 'Id_Consorcio','Path'])
for _, row in directory.iterrows():
  URL = row['URL']
  Folder = row['Folder']
  Files = row['Files']
  consorcio = row['Id_Consorcio']
  for file in Files:
    if mes_proceso in file:
      link = URL
      carpeta = Folder
      archivo = file
      consorcio_2 = consorcio
      directorio = os.path.join(Folder,file)
      archivos_despivoteados.loc[len(archivos_despivoteados)] = [link,carpeta,archivo, consorcio_2, directorio]

archivos_despivoteados['Folder'] = archivos_despivoteados.apply(lambda row: row['Folder'].replace(row['URL'],'').strip(),axis = 1)
df = archivos_despivoteados
df = df[df['Path'].str.endswith('.pdf')]

In [ ]:
#Agrupo los archivos en las categorias elementales, analizo si hay varios de la misma categoria y armo un nombre de archivo
conditions = [
    df['Folder'].str.contains('/1 HABERES, CARGAS SOCIALES Y SINDICALES',case=False),
    df['Folder'].str.contains('/1 SERVICIO TERCERIZADO DE LIMPIEZA - HABERES',case=False),
    df['Folder'].str.contains('/2 SERVICIOS PUBLICOS',case=False),
    df['Folder'].str.contains('/3 ABONOS DE SERVICIOS',case=False),
    df['Folder'].str.contains('/4 MANT. DE PARTES COMUNES',case=False),
    df['Folder'].str.contains('/5 REPARAC. DE UNIDADES PARTICULARES',case=False),
    df['Folder'].str.contains('/6 BANCO',case=False),
    df['Folder'].str.contains('/7 GASTOS INSUMOS DE LIMPIEZA',case=False),
    df['Folder'].str.contains('/8 ADMINISTRACION',case=False),
    df['Folder'].str.contains('/9 SEGUROS',case=False),
    df['Folder'].str.contains('/10 OTROS',case=False),
    df['Folder'].str.contains('/11 GASTOS EXTRAORDINARIOS',case=False),
    df['Folder'].str.contains('/REGISTRO LIQ. EXPENSAS',case=False)
    ]
choices = ['Haberes, CS y Sind.','Serv. Terc. Limpieza','Servicios Publicos','Abono Servicio',
           'Mantenimiento Partes Comunes','Reparacion Unidades Part.','Banco','Insumos Limpieza','No Pasa',
           'Seguros','Otros','Gastos Extraordinarios','Expensas']

df['Agrupacion'] = np.select(conditions, choices,default = 'No Pasa')

df3 = df.groupby(['Agrupacion','Id_Consorcio'])['Folder'].count().reset_index()
df3.columns = ['Agrup','Id','Repeticiones']

archivos_proceso = df.merge(df3, left_on =['Agrupacion','Id_Consorcio'],right_on = ['Agrup','Id'])
archivos_proceso.drop(['Agrup','Id'], axis=1, inplace = True)
archivos_proceso['nro_repeticion'] = archivos_proceso.groupby(['Agrupacion','Id_Consorcio']).cumcount()+1
archivos_proceso['nombre_archivo'] = (archivos_proceso['Agrupacion'] + '_' + mes_proceso[:4] + '_' +  mes_proceso[-2:] + '_' +
                                      archivos_proceso['Id_Consorcio'].map(str) + '_' +  archivos_proceso.nro_repeticion.map(str) + '.pdf')
archivos_proceso['carpeta_principal'] = archivos_proceso['Folder'].str.split('/').str[1]

In [ ]:
#Cruzar con el registro de archivos ya subidos
df_filtrado = archivos_proceso[archivos_proceso['Agrupacion'] != 'No Pasa']
df_filtrado['subido_simple_sol'] = pd.Timestamp.now(tz= 'America/Argentina/Buenos_Aires').replace(tzinfo=None)
#si cambia de usuario hay que cambiar estas carpetas
archivos_ya_subidos = pd.read_excel(UPLOAD_LOG_PATH)
path_subidos = pd.DataFrame(archivos_ya_subidos['Path'])
path_subidos.columns = ['ya_subido']
df_filtrado_2 = pd.merge(df_filtrado,path_subidos, how = 'left',right_on = 'ya_subido',left_on='Path')
df_filtrado_2['ya_subido'] = df_filtrado_2['ya_subido'].notnull()
df_filtrado_2 = df_filtrado_2[df_filtrado_2['ya_subido']==False].reset_index()
df_filtrado_2.drop(['ya_subido','index'], axis=1, inplace = True)


In [ ]:
!pip install paramiko
# !pip3 install paramiko update

import paramiko

def upload_file_sftp(hostname, port, username, private_key_path, remote_path, local_path):
    try:
        # Create a new SSH client object
        ssh_client = paramiko.SSHClient()

        # Automatically add the server's host key
        ssh_client.load_system_host_keys()
    ssh_client.set_missing_host_key_policy(paramiko.RejectPolicy())

        # Load the private key for authentication
        private_key = paramiko.RSAKey.from_private_key_file(private_key_path)

        # Connect to the SFTP server
        ssh_client.connect(hostname, port=port, username=username, pkey=private_key)

        # Create an SFTP client from the SSH client
        sftp_client = ssh_client.open_sftp()

        # Change to the remote directory
        sftp_client.chdir(remote_path)

        # Upload the file
        sftp_client.put(local_path,  remote_path + '/' + filename)

        # print(f"File {local_path} uploaded successfully to {remote_path}")

        # Close the SFTP session and the SSH connection
        sftp_client.close()
        ssh_client.close()

    except Exception as e:
        print(f"Error: {e}")
        raise

# Fill in your connection details
hostname = os.environ['SFTP_HOST']
port = 22  # Default SFTP port is 22
username = os.environ['SFTP_USERNAME']
#si cambia de usuario hay que cambiar estas carpetas
private_key_path = os.environ['SFTP_PRIVATE_KEY_PATH']


In [ ]:
fin_de_archivo = len(path_subidos)+1
for n in df_filtrado_2.index:
  local_path = df_filtrado_2['Path'][n]
  filename = df_filtrado_2['nombre_archivo'][n]
  fila = df_filtrado_2[n:n+1]
  if df_filtrado_2['Agrupacion'][n] == 'Expensas':
    remote_path = os.environ['SFTP_EXPENSES_FOLDER']
  else:
    remote_path = os.environ['SFTP_DOCUMENTS_FOLDER']

  upload_file_sftp(hostname, port, username, private_key_path, remote_path, local_path)
  #si cambia de usuario hay que cambiar estas carpetas
  with pd.ExcelWriter(UPLOAD_LOG_PATH,
                    if_sheet_exists = 'overlay', mode='a') as writer:
    fila.to_excel(writer, sheet_name ='Sheet1', index = False , header = False, startrow=fin_de_archivo)
  fin_de_archivo = fin_de_archivo + 1
  print(filename)
print("Fecha y Hora al terminar el proceso:", pd.Timestamp.now(tz= 'America/Argentina/Buenos_Aires').floor('s').replace(tzinfo=None))
  # if n == 5:
  #   break


In [ ]:
df_filtrado_2